In [ ]:
%pip install transformers[torch] pandas openpyxl scikit-learn

: 

In [ ]:
import pandas as pd

df = pd.read_csv('labeled_data.csv')
print("5 dòng dữ liệu đầu tiên:")
print(df.head())
print("Thống kê số lượng bài báo mỗi chuyên mục:")
print(df['category'].value_counts())

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "vinai/phobert-base-v2"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=8)
    print("✅ Tải PhoBERT model và tokenizer thành công!")
except Exception as e:
    print(f"❌ Lỗi khi tải model: {e}")

In [ ]:
# Lấy danh sách chuyên mục duy nhất
unique_categories = df['category'].unique().tolist()

# Bản đồ chuyển đổi
label2id = {label: i for i, label in enumerate(unique_categories)}
id2label = {i: label for i, label in enumerate(unique_categories)}

# Tạo cột nhãn số
df['label'] = df['category'].map(label2id)
print("label2id:", label2id)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=256)

In [ ]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset  = NewsDataset(test_encodings,  test_labels)

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
!pip install -U transformers

In [ ]:
!pip uninstall -y transformers tokenizers accelerate

In [ ]:
!pip install -U "transformers[torch]==4.44.0" accelerate datasets

In [ ]:
import transformers
print("transformers version:", transformers.__version__)

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.save_model("my-finetuned-phobert")
tokenizer.save_pretrained("my-finetuned-phobert")

In [ ]:
metrics = trainer.evaluate(test_dataset)
print("🎯 Độ chính xác (Accuracy) trên tập test:", metrics["eval_accuracy"])
print("📊 F1-score:", metrics["eval_f1"])

In [ ]:
from google.colab import files
!zip -r my-finetuned-phobert.zip my-finetuned-phobert
files.download("my-finetuned-phobert.zip")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Đường dẫn lưu trên Drive
base_path = "/content/drive/MyDrive/ai_notebooks"

# Tạo các thư mục con
os.makedirs(f"{base_path}/models", exist_ok=True)
os.makedirs(f"{base_path}/logs", exist_ok=True)
os.makedirs(f"{base_path}/notebooks", exist_ok=True)

# Tạo file README.md mẫu
readme_content = """# 📘 ai_notebooks

Thư mục này lưu trữ kết quả huấn luyện mô hình AI (PhoBERT fine-tuning).

## Cấu trúc thư mục
- **models/** : chứa các mô hình đã huấn luyện (.zip hoặc folder)
- **logs/** : chứa file log hoặc csv ghi kết quả
- **notebooks/** : chứa các notebook (.ipynb)
- **README.md** : mô tả thông tin huấn luyện, thông số, kết quả và nhận xét

## Ghi chú:
Mỗi lần huấn luyện, ghi lại:
- Thời gian train
- Thông số (epochs, batch_size, lr, max_length)
- Kết quả (Accuracy, F1)
- Nhận xét, lỗi gặp phải, cải tiến
"""

with open(f"{base_path}/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ Đã tạo thư mục 'ai_notebooks/' trong Google Drive thành công!")
print(f"📂 Đường dẫn: {base_path}")


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()